In [ ]:
# ============================================================
# SECTION 12 : FINE-TUNING + OPTUNA — M5 CNN+ViT
# ============================================================

!pip install -q optuna
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Fonction de Fine-Tuning ───────────────────────────────────
def degeler_couches(model, nom, nb_couches=2):
    """
    Degele les dernieres couches d'un modele pre-entraine.
    - nb_couches=0 : seul le classifier est entraine
    - nb_couches=1 : + dernier bloc de convolution
    - nb_couches=2 : + 2 derniers blocs (recommande)
    """
    for param in model.parameters():
        param.requires_grad = False

    # Degeler le classifier (toujours)
    if hasattr(model, 'fc'):
        for param in model.fc.parameters():
            param.requires_grad = True
    elif hasattr(model, 'classifier'):
        for param in model.classifier.parameters():
            param.requires_grad = True

    if nb_couches == 0:
        return model

    if nom == "M5 CNN+ViT":
        # Degeler le Transformer complet
        for param in model.transformer.parameters():
            param.requires_grad = True
        # Degeler les derniers blocs du backbone selon nb_couches
        backbone_layers = list(model.backbone.children())
        for layer in backbone_layers[-nb_couches:]:
            for param in layer.parameters():
                param.requires_grad = True
        # Degeler la projection et le CLS token
        for param in model.proj.parameters():
            param.requires_grad = True
        model.cls_token.requires_grad = True
        model.pos_embed.requires_grad = True

    total    = sum(p.numel() for p in model.parameters())
    entraine = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Fine-tuning {nb_couches} blocs : {entraine:,} / {total:,} parametres ({entraine/total*100:.1f}%)")
    return model

# ── Evaluation rapide pour Optuna ────────────────────────────
def evaluer_rapide_optuna(model, val_loader):
    """Evalue le Recall PNEUMONIA sur le val set."""
    model.eval()
    labels_all, preds_all = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            labels_all.extend(labels.cpu().numpy())
            preds_all.extend(preds.cpu().numpy())
    return recall_score(labels_all, preds_all, pos_label=1, zero_division=0)

# ── Objectif Optuna ───────────────────────────────────────────
def objectif_optuna(trial, base_model_fn, nom):
    """
    Optuna maximise le Recall PNEUMONIA sur le val set.
    TPE Sampler apprend des meilleurs trials precedents.
    Entrainement court : 5 epochs pour comparer rapidement.
    """
    lr           = trial.suggest_float("lr",           1e-5, 1e-3, log=True)
    nb_couches   = trial.suggest_int("nb_couches",     0,    2)
    dropout      = trial.suggest_float("dropout",      0.2,  0.6)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    model_trial = base_model_fn(dropout)
    model_trial = degeler_couches(model_trial, nom, nb_couches)
    model_trial = model_trial.to(device)

    criterion = FocalLoss(alpha=1.0, gamma=2.0)
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model_trial.parameters()),
        lr=lr, weight_decay=weight_decay
    )

    for epoch in range(5):
        fixer_seed(epoch)
        model_trial.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model_trial(imgs), labels)
            loss.backward()
            optimizer.step()

    recall = evaluer_rapide_optuna(model_trial, val_loader)
    del model_trial
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return recall

# ── Lancer Optuna sur M5 CNN+ViT ─────────────────────────────
print("="*65)
print("OPTUNA — Optimisation hyperparametres Fine-Tuning CNN+ViT")
print("20 trials x 5 epochs | TPE Sampler")
print("="*65)

def creer_cnn_vit(dropout=0.3):
    """Recrée une instance propre de HybrideCNNViT."""
    return HybrideCNNViT(nb_classes=2, d_model=512, nhead=8,
                         num_layers=2, dropout=dropout)

sampler = TPESampler(seed=42)
etude   = optuna.create_study(direction="maximize", sampler=sampler)
etude.optimize(
    lambda trial: objectif_optuna(trial, creer_cnn_vit, "M5 CNN+ViT"),
    n_trials=20,
    show_progress_bar=True
)

# ── Resultats ─────────────────────────────────────────────────
meilleur = etude.best_trial
print(f"\nMeilleur trial : #{meilleur.number}")
print(f"Meilleur Recall PNEUMONIA (val) : {meilleur.value:.4f}")
print("\nHyperparametres optimaux :")
for k, v in meilleur.params.items():
    print(f"  {k:<20} : {v}")

BEST_LR           = meilleur.params["lr"]
BEST_NB_COUCHES   = meilleur.params["nb_couches"]
BEST_DROPOUT      = meilleur.params["dropout"]
BEST_WEIGHT_DECAY = meilleur.params["weight_decay"]

# ── Visualisation des trials ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Optuna — Exploration des Hyperparametres (CNN+ViT Fine-Tuning)",
             fontsize=13, fontweight="bold")

trials_ok = [t for t in etude.trials if t.value is not None]
vals      = [t.value for t in trials_ok]
lr_v      = [t.params["lr"]           for t in trials_ok]
nb_c_v    = [t.params["nb_couches"]   for t in trials_ok]
drop_v    = [t.params["dropout"]      for t in trials_ok]
wd_v      = [t.params["weight_decay"] for t in trials_ok]

sc = axes[0][0].scatter(lr_v, vals, c=vals, cmap="RdYlGn", alpha=0.8, s=80)
axes[0][0].set_xscale("log"); axes[0][0].set_xlabel("Learning Rate")
axes[0][0].set_ylabel("Recall PNEUMONIA")
axes[0][0].set_title("LR vs Recall")
axes[0][0].axhline(y=meilleur.value, color="red", linestyle="--", alpha=0.5)
axes[0][0].grid(True, alpha=0.3); plt.colorbar(sc, ax=axes[0][0])

axes[0][1].scatter(nb_c_v, vals, c=vals, cmap="RdYlGn", alpha=0.8, s=80)
axes[0][1].set_xlabel("Nombre de blocs degelez")
axes[0][1].set_ylabel("Recall PNEUMONIA")
axes[0][1].set_title("Blocs Fine-Tuning vs Recall")
axes[0][1].axhline(y=meilleur.value, color="red", linestyle="--", alpha=0.5)
axes[0][1].grid(True, alpha=0.3)

axes[1][0].scatter(drop_v, vals, c=vals, cmap="RdYlGn", alpha=0.8, s=80)
axes[1][0].set_xlabel("Dropout")
axes[1][0].set_ylabel("Recall PNEUMONIA")
axes[1][0].set_title("Dropout vs Recall")
axes[1][0].axhline(y=meilleur.value, color="red", linestyle="--", alpha=0.5)
axes[1][0].grid(True, alpha=0.3)

axes[1][1].scatter(wd_v, vals, c=vals, cmap="RdYlGn", alpha=0.8, s=80)
axes[1][1].set_xscale("log"); axes[1][1].set_xlabel("Weight Decay")
axes[1][1].set_ylabel("Recall PNEUMONIA")
axes[1][1].set_title("Weight Decay vs Recall")
axes[1][1].axhline(y=meilleur.value, color="red", linestyle="--", alpha=0.5)
axes[1][1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print("Zones vertes = hyperparametres prometteurs.")
print("Optuna concentre sa recherche sur ces zones au fil des trials.")